# *Data Describtion*



In [ ]:
import pandas as pd
from autoimpute.imputations import SingleImputer


In [ ]:
df=pd.read_csv(r"Telco_Churn_Data.csv")
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1.0,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34.0,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2.0,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45.0,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2.0,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7093 entries, 0 to 7092
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7093 non-null   object 
 1   gender            7093 non-null   object 
 2   SeniorCitizen     7093 non-null   int64  
 3   Partner           7093 non-null   object 
 4   Dependents        7093 non-null   object 
 5   tenure            7073 non-null   float64
 6   PhoneService      6940 non-null   object 
 7   MultipleLines     6927 non-null   object 
 8   InternetService   7093 non-null   object 
 9   OnlineSecurity    6951 non-null   object 
 10  OnlineBackup      6949 non-null   object 
 11  DeviceProtection  7093 non-null   object 
 12  TechSupport       6940 non-null   object 
 13  StreamingTV       7093 non-null   object 
 14  StreamingMovies   7093 non-null   object 
 15  Contract          7078 non-null   object 
 16  PaperlessBilling  6957 non-null   object 


# Data Cleaning


In [ ]:
# remove CustomerID column
df.drop('customerID', axis=1, inplace=True) # because it is not useful for prediction

missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
quality_report = pd.DataFrame({
'Missing Count' : missing,
'Missing %'     : missing_pct
}).query('`Missing Count` > 0')

print("Missing Values:")
print(quality_report.to_string())
print(f"Duplicate rows: {df.duplicated().sum()}")

Missing Values:
                  Missing Count  Missing %
tenure                       20       0.28
PhoneService                153       2.16
MultipleLines               166       2.34
OnlineSecurity              142       2.00
OnlineBackup                144       2.03
TechSupport                 153       2.16
Contract                     15       0.21
PaperlessBilling            136       1.92
PaymentMethod               178       2.51
MonthlyCharges              198       2.79
TotalCharges                151       2.13
Duplicate rows: 56


## *Remove duplicated values*

In [ ]:
before = len(df)
df.drop_duplicates(inplace=True)
after  = len(df)

print(f"Rows before : {before}")
print(f"Rows removed: {before - after}")
print(f"Rows after  : {after}")

Rows before : 7093
Rows removed: 56
Rows after  : 7037


## *convert type of TotalCharges*



In [ ]:
# type corrections
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
for col in df.columns:
    print(f"{col}: {df[col].unique()} unique values")


gender: ['Female' 'Male'] unique values
SeniorCitizen: [0 1] unique values
Partner: ['Yes' 'No'] unique values
Dependents: ['No' 'Yes'] unique values
tenure: [ 1. 34.  2. 45.  8. 22. 10. 28. 62. 13. 16. 58. 49. 25. 69. 52. 71. 21.
 12. 30. 47. 72. 17. 27.  5. 46. 11. 70. 63. 43. 15. 60. 18. 66.  9.  3.
 31. 50. 64. 56.  7. 42. 35. 48. 29. 65. 38. 68. 32. 55. 37. 36. 41.  6.
  4. 33. 67. 23. 57. 61. 14. 20. 53. nan 40. 59. 24. 44. 19. 54. 51. 26.
  0. 39.] unique values
PhoneService: ['No' 'Yes' nan] unique values
MultipleLines: ['No phone service' 'No' 'Yes' nan] unique values
InternetService: ['DSL' 'Fiber optic' 'No'] unique values
OnlineSecurity: ['No' 'Yes' 'No internet service' nan] unique values
OnlineBackup: ['Yes' nan 'No' 'No internet service'] unique values
DeviceProtection: ['No' 'Yes' 'No internet service'] unique values
TechSupport: ['No' 'Yes' 'No internet service' nan] unique values
StreamingTV: ['No' 'Yes' 'No internet service'] unique values
StreamingMovies: ['No' 'Yes

## *Replace inconsistent values in some columns*


In [ ]:
df.replace('No internet service', 'No', inplace=True)
df['PaymentMethod'] = df['PaymentMethod'].str.replace(r'\s*\(automatic\)', '', regex=True)
df['Contract'] = df['Contract'].replace('Month-to-month', 'Monthly')
df['MultipleLines']=df['MultipleLines'].replace('No phone service', 'No')
# print values of each column
for col in df.columns:
    print(f"{col}: {df[col].unique()} unique values")

gender: ['Female' 'Male'] unique values
SeniorCitizen: [0 1] unique values
Partner: ['Yes' 'No'] unique values
Dependents: ['No' 'Yes'] unique values
tenure: [ 1. 34.  2. 45.  8. 22. 10. 28. 62. 13. 16. 58. 49. 25. 69. 52. 71. 21.
 12. 30. 47. 72. 17. 27.  5. 46. 11. 70. 63. 43. 15. 60. 18. 66.  9.  3.
 31. 50. 64. 56.  7. 42. 35. 48. 29. 65. 38. 68. 32. 55. 37. 36. 41.  6.
  4. 33. 67. 23. 57. 61. 14. 20. 53. nan 40. 59. 24. 44. 19. 54. 51. 26.
  0. 39.] unique values
PhoneService: ['No' 'Yes' nan] unique values
MultipleLines: ['No' 'Yes' nan] unique values
InternetService: ['DSL' 'Fiber optic' 'No'] unique values
OnlineSecurity: ['No' 'Yes' nan] unique values
OnlineBackup: ['Yes' nan 'No'] unique values
DeviceProtection: ['No' 'Yes'] unique values
TechSupport: ['No' 'Yes' nan] unique values
StreamingTV: ['No' 'Yes'] unique values
StreamingMovies: ['No' 'Yes'] unique values
Contract: ['Monthly' 'One year' 'Two year' nan] unique values
PaperlessBilling: ['Yes' 'No' nan] unique values
P

## *Handle missing values*

In [ ]:
missing_columns_category = [col for col in quality_report.index if df[col].dtype == 'object']
# hanle missing values
strategy_dict = {col: 'mode' for col in missing_columns_category}
df[missing_columns_category] = SingleImputer(
    strategy=strategy_dict,
    seed=42
).fit_transform(df[missing_columns_category])

# fill missing values in numerical columns using median per group
df['MonthlyCharges'] = df.groupby('InternetService')['MonthlyCharges'].transform(lambda x: x.fillna(x.median()))
df['tenure'] = df.groupby('Contract')['tenure'].transform(lambda x: x.fillna(x.median()))
df['tenure'] = df['tenure'].round().astype(int)
#spacial case fot total charges because it is related to tenure and monthley charges (if tenure is 0 then total charges eqal monthley charges)
mask_tc = df['TotalCharges'].isnull() & (df['tenure'] == 0)
df.loc[mask_tc, 'TotalCharges'] = df.loc[mask_tc, 'MonthlyCharges']

df['TotalCharges'] = df['TotalCharges'].fillna(
    df['tenure'] * df['MonthlyCharges'])

print("Number of missing values:\n", df.isnull().sum())
df.head()


Number of missing values:
 gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No,DSL,No,Yes,No,No,No,No,Monthly,Yes,Electronic check,29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Monthly,Yes,Mailed check,53.85,108.15,Yes
3,Male,0,No,No,45,No,No,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer,42.30,1840.75,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Monthly,Yes,Electronic check,70.70,151.65,Yes


In [ ]:
pivot = pd.pivot_table(
df,
values   = 'MonthlyCharges',
index    = 'Contract',
columns  = 'InternetService',
aggfunc  = 'mean'
).round(2)

print("Average Monthly Charges by Contract x Internet Service:")
print(pivot.head())

# Churn Rate per Contract type
churn_pivot = pd.pivot_table(
df,
values  = 'Churn',
index   = 'Contract',
aggfunc = lambda x: (x == 'Yes').sum() / len(x) * 100
).round(2)
churn_pivot.columns = ['Churn Rate %']
print(r"\nChurn Rate % by Contract Type:")
print(churn_pivot.head())

Average Monthly Charges by Contract x Internet Service:
InternetService    DSL  Fiber optic     No
Contract                                  
Monthly          50.51        87.26  20.39
One year         61.14        98.64  20.82
Two year         70.16       104.19  21.71
\nChurn Rate % by Contract Type:
          Churn Rate %
Contract              
Monthly          42.64
One year         11.10
Two year          2.83


# *Encoding*

In [ ]:
# Binary
binary_cols = [col for col in df.columns if df[col].nunique() == 2 and df[col].dtype == 'object']
for col in binary_cols:
    df[col] = df[col].map({'Yes':1, 'No':0, 'Male':1, 'Female':0})
# Ordinal data
df['Contract'] = df['Contract'].map({
    'Monthly': 0,
    'One year': 1,
    'Two year': 2
})

# One Hot
df= pd.get_dummies(df, columns=[
    'InternetService',
    'PaymentMethod',
    'MultipleLines'
],dtype=int)


df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,...,Churn,InternetService_DSL,InternetService_Fiber optic,InternetService_No,PaymentMethod_Bank transfer,PaymentMethod_Credit card,PaymentMethod_Electronic check,PaymentMethod_Mailed check,MultipleLines_0,MultipleLines_1
0,0,0,1,0,1,0,0,1,0,0,...,0,1,0,0,0,0,1,0,1,0
1,1,0,0,0,34,1,1,0,1,0,...,0,1,0,0,0,0,0,1,1,0
2,1,0,0,0,2,1,1,1,0,0,...,1,1,0,0,0,0,0,1,1,0
3,1,0,0,0,45,0,1,0,1,1,...,0,1,0,0,1,0,0,0,1,0
4,0,0,0,0,2,1,0,0,0,0,...,1,0,1,0,0,0,1,0,1,0


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7037 entries, 0 to 7091
Data columns (total 26 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   gender                          7037 non-null   int64  
 1   SeniorCitizen                   7037 non-null   int64  
 2   Partner                         7037 non-null   int64  
 3   Dependents                      7037 non-null   int64  
 4   tenure                          7037 non-null   int64  
 5   PhoneService                    7037 non-null   int64  
 6   OnlineSecurity                  7037 non-null   int64  
 7   OnlineBackup                    7037 non-null   int64  
 8   DeviceProtection                7037 non-null   int64  
 9   TechSupport                     7037 non-null   int64  
 10  StreamingTV                     7037 non-null   int64  
 11  StreamingMovies                 7037 non-null   int64  
 12  Contract                        7037 no

In [ ]:
# print values of each column
for col in df.columns:
    print(f"{col}: {df[col].unique()} unique values")

gender: [0 1] unique values
SeniorCitizen: [0 1] unique values
Partner: [1 0] unique values
Dependents: [0 1] unique values
tenure: [ 1 34  2 45  8 22 10 28 62 13 16 58 49 25 69 52 71 21 12 30 47 72 17 27
  5 46 11 70 63 43 15 60 18 66  9  3 31 50 64 56  7 42 35 48 29 65 38 68
 32 55 37 36 41  6  4 33 67 23 57 61 14 20 53 40 59 24 44 19 54 51 26  0
 39] unique values
PhoneService: [0 1] unique values
OnlineSecurity: [0 1] unique values
OnlineBackup: [1 0] unique values
DeviceProtection: [0 1] unique values
TechSupport: [0 1] unique values
StreamingTV: [0 1] unique values
StreamingMovies: [0 1] unique values
Contract: [0 1 2] unique values
PaperlessBilling: [1 0] unique values
MonthlyCharges: [29.85 56.95 53.85 ... 63.1  44.2  78.7 ] unique values
TotalCharges: [  29.85 1889.5   108.15 ...  346.45  306.6  6844.5 ] unique values
Churn: [0 1] unique values
InternetService_DSL: [1 0] unique values
InternetService_Fiber optic: [0 1] unique values
InternetService_No: [0 1] unique values
Paym

# Feature Extraction

In [ ]:
service_cols = [
    'PhoneService', 'OnlineSecurity', 'OnlineBackup',
    'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies'
]

df['TotalServices'] = df[service_cols].sum(axis=1)
print(df[['TotalServices']].head())

   TotalServices
0              1
1              3
2              3
3              3
4              1


In [ ]:
df['AvgMonthlySpend'] = df['TotalCharges'] / (df['tenure'] + 1)
print(df[['AvgMonthlySpend']].head())

   AvgMonthlySpend
0        14.925000
1        53.985714
2        36.050000
3        40.016304
4        50.550000


In [ ]:
df['TenureGroup'] = pd.cut(
    df['tenure'],
    bins=[-1, 12, 36, 100],
    labels=[0, 1, 2]
).astype(int)
print(df[['TenureGroup']].head())

   TenureGroup
0            0
1            1
2            0
3            2
4            0


In [ ]:
df['HighValueCustomer'] = (df['MonthlyCharges'] > df['MonthlyCharges'].median()).astype(int)
print(df[['HighValueCustomer']].head())

   HighValueCustomer
0                  0
1                  0
2                  0
3                  0
4                  1


In [ ]:
df['ContractRisk'] = df['Contract'].apply(lambda x: 2 if x == 0 else (1 if x == 1 else 0))
print(df[['ContractRisk']].head())

   ContractRisk
0             2
1             1
2             2
3             1
4             2


In [ ]:
df['Charge_Tenure_Interaction'] = df['MonthlyCharges'] * df['tenure']
print(df[['Charge_Tenure_Interaction']].head())

   Charge_Tenure_Interaction
0                      29.85
1                    1936.30
2                     107.70
3                    1903.50
4                     141.40


In [ ]:
import numpy as np

df['LogMonthlyCharges'] = np.log1p(df['MonthlyCharges'])
df['LogTotalCharges'] = np.log1p(df['TotalCharges'])

print(df[['MonthlyCharges', 'LogMonthlyCharges']].head())
print(df[['TotalCharges', 'LogTotalCharges']].head())

   MonthlyCharges  LogMonthlyCharges
0           29.85           3.429137
1           56.95           4.059581
2           53.85           4.004602
3           42.30           3.768153
4           70.70           4.272491
   TotalCharges  LogTotalCharges
0         29.85         3.429137
1       1889.50         7.544597
2        108.15         4.692723
3       1840.75         7.518471
4        151.65         5.028148


In [ ]:
df['ChargesPerTenure'] = df['TotalCharges'] / (df['tenure'] + 1)
df['MonthlyToTotalRatio'] = df['MonthlyCharges'] / (df['TotalCharges'] + 1)

print(df[['ChargesPerTenure']].head())
print(df[['MonthlyToTotalRatio']].head())

   ChargesPerTenure
0         14.925000
1         53.985714
2         36.050000
3         40.016304
4         50.550000
   MonthlyToTotalRatio
0             0.967585
1             0.030124
2             0.493358
3             0.022967
4             0.463151


In [ ]:
df['ExpectedVsActualCharges'] = df['MonthlyCharges'] * df['tenure'] - df['TotalCharges']

print(df[['ExpectedVsActualCharges']].head())

   ExpectedVsActualCharges
0                     0.00
1                    46.80
2                    -0.45
3                    62.75
4                   -10.25


In [ ]:
df['ServiceEfficiency'] = df['TotalCharges'] / (df['TotalServices'] + 1)

print(df[['ServiceEfficiency']].head())

   ServiceEfficiency
0            14.9250
1           472.3750
2            27.0375
3           460.1875
4            75.8250


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

X = df.drop('Churn', axis=1)
y = df['Churn']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=5)
X_pca = pca.fit_transform(X_scaled)

pca_df = pd.DataFrame(X_pca, columns=['PC1','PC2','PC3','PC4','PC5'])

pca_df['Churn'] = y.values

pca_df.head()

,PC1,PC2,PC3,PC4,PC5,Churn
0,-4.828710,-1.435366,2.699284,-1.732644,-0.111019,0
1,-1.116337,1.352321,1.771930,0.756932,-0.668057,0
2,-3.375597,-1.204144,2.499306,1.178334,-1.230570,1
3,-0.768264,2.627825,3.179457,-1.746859,-0.796868,0
4,-2.530791,-3.564568,-0.440610,-0.131715,0.881365,1


In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
X = df.drop('Churn', axis=1)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df['CustomerCluster'] = kmeans.fit_predict(X_scaled)

df[['CustomerCluster']].head()

,CustomerCluster
0,1
1,1
2,1
3,1
4,1


In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.preprocessing import StandardScaler

num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

X_num = df[num_cols]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_num)

poly = PolynomialFeatures(degree=2, include_bias=False)

X_poly = poly.fit_transform(X_scaled)

poly_df = pd.DataFrame(X_poly, columns=poly.get_feature_names_out(num_cols))

poly_df.head()

,tenure,MonthlyCharges,TotalCharges,tenure^2,tenure MonthlyCharges,tenure TotalCharges,MonthlyCharges^2,MonthlyCharges TotalCharges,TotalCharges^2
0,-1.280885,-1.164399,-0.994707,1.640667,1.491461,1.274105,1.355824,1.158236,0.989442
1,0.063490,-0.262304,-0.174719,0.004031,-0.016654,-0.011093,0.068803,0.045830,0.030527
2,-1.240146,-0.365495,-0.960182,1.537963,0.453268,1.190766,0.133587,0.350942,0.921949
3,0.511615,-0.749968,-0.196215,0.261750,-0.383695,-0.100387,0.562451,0.147155,0.038500
4,-1.240146,0.195401,-0.941001,1.537963,-0.242326,1.166979,0.038182,-0.183873,0.885483


# Feature Selection

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.feature_selection import mutual_info_classif
from sklearn.ensemble import RandomForestClassifier

X = df.drop('Churn', axis=1)
y = df['Churn']
# Train & Test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# METHOD 1: FILTER METHODS (Chi-Square + Mutual Info)

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Chi square
chi_selector = SelectKBest(score_func=chi2, k=10)
chi_selector.fit(X_train_scaled, y_train)

chi_scores = pd.DataFrame({
    'Feature': X_train.columns,
    'Score': chi_selector.scores_
}).sort_values(by='Score', ascending=False)

print("Chi_Square Top Features:")
print( chi_scores.head(10))

chi_features = X_train.columns[chi_selector.get_support()]

Chi_Square Top Features:
                           Feature       Score
12                        Contract  382.128874
21  PaymentMethod_Electronic check  277.197209
17     InternetService_Fiber optic  261.300880
34             MonthlyToTotalRatio  204.163819
29                    ContractRisk  199.179143
18              InternetService_No  189.557056
27                     TenureGroup  184.112858
4                           tenure  156.584060
1                    SeniorCitizen  107.670927
28               HighValueCustomer  100.208292


In [ ]:
# Mutual Information
mi_selector = SelectKBest(score_func=mutual_info_classif, k=10)
mi_selector.fit(X_train, y_train)

mi_scores = pd.DataFrame({
    'Feature': X_train.columns,
    'Score': mi_selector.scores_
}).sort_values(by='Score', ascending=False)

print("Mutual Information Top Features:")
print(mi_scores.head(10))

mi_features = X_train.columns[mi_selector.get_support()]

Mutual Information Top Features:
                        Feature     Score
29                 ContractRisk  0.095430
12                     Contract  0.090568
34          MonthlyToTotalRatio  0.083651
37              CustomerCluster  0.071652
4                        tenure  0.069577
36            ServiceEfficiency  0.058395
17  InternetService_Fiber optic  0.057749
27                  TenureGroup  0.057202
14               MonthlyCharges  0.050486
31            LogMonthlyCharges  0.049274


In [ ]:
# Comparing chi square & mutual info
chi_set = set(chi_features)
mi_set = set(mi_features)

print("Common Features:", chi_set & mi_set)
print("Chi only:", chi_set - mi_set)
print("MI only:", mi_set - chi_set)

Common Features: {'MonthlyToTotalRatio', 'ContractRisk', 'InternetService_Fiber optic', 'Contract', 'tenure', 'TenureGroup'}
Chi only: {'SeniorCitizen', 'PaymentMethod_Electronic check', 'HighValueCustomer', 'InternetService_No'}
MI only: {'LogMonthlyCharges', 'CustomerCluster', 'MonthlyCharges', 'ServiceEfficiency'}


# METHOD 2: EMBEDDED (Random Forest Importance)

In [ ]:
# Random Forest Importance
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)

importances = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': rf.feature_importances_
}).sort_values(by='Importance', ascending=False)

print("Top Embedded Features:")
print(importances.head(10))

Top Embedded Features:
                      Feature  Importance
34        MonthlyToTotalRatio    0.088264
36          ServiceEfficiency    0.067321
14             MonthlyCharges    0.066111
31          LogMonthlyCharges    0.061584
32            LogTotalCharges    0.059762
26            AvgMonthlySpend    0.058728
15               TotalCharges    0.058222
33           ChargesPerTenure    0.057418
30  Charge_Tenure_Interaction    0.055704
35    ExpectedVsActualCharges    0.052043


In [ ]:
embedded_features = importances.head(10)['Feature'].values
print(list(embedded_features))

['MonthlyToTotalRatio', 'ServiceEfficiency', 'MonthlyCharges', 'LogMonthlyCharges', 'LogTotalCharges', 'AvgMonthlySpend', 'TotalCharges', 'ChargesPerTenure', 'Charge_Tenure_Interaction', 'ExpectedVsActualCharges']


# Comparison between Filter and Embedded

In [ ]:
filter_features = set(mi_features)   #  or chi_features
embedded_features_set = set(embedded_features)

print("Common Features:")
print(filter_features & embedded_features_set)

print("\nOnly Filter Methods:")
print(filter_features - embedded_features_set)

print("\nOnly Embedded Method:")
print(embedded_features_set - filter_features)

Common Features:
{'LogMonthlyCharges', 'MonthlyCharges', 'MonthlyToTotalRatio', 'ServiceEfficiency'}

Only Filter Methods:
{'CustomerCluster', 'ContractRisk', 'InternetService_Fiber optic', 'Contract', 'tenure', 'TenureGroup'}

Only Embedded Method:
{'AvgMonthlySpend', 'LogTotalCharges', 'ChargesPerTenure', 'ExpectedVsActualCharges', 'TotalCharges', 'Charge_Tenure_Interaction'}


# FINAL SELECTED DATASET

In [ ]:
final_features = list(filter_features & embedded_features_set)

X_train_selected = X_train[final_features]
X_test_selected = X_test[final_features]

print("Final Selected Features:", final_features)
print("Shape:", X_train_selected.shape)
X_train_selected.head()

Final Selected Features: ['LogMonthlyCharges', 'MonthlyCharges', 'MonthlyToTotalRatio', 'ServiceEfficiency']
Shape: (4925, 4)


,LogMonthlyCharges,MonthlyCharges,MonthlyToTotalRatio,ServiceEfficiency
4790,4.452602,84.85,0.032208,877.800000
6964,4.048301,56.30,0.020240,556.120000
3097,3.063391,20.40,0.464692,21.450000
6546,3.954124,51.15,0.040064,425.233333
2871,4.467057,86.10,0.069629,411.850000
